# 12a — Validation Evaluation

This notebook performs Week 6 validation using the **saved fitted Week 5 tuned pipelines** created in Notebook 11b.

### Week 5 consistency approach
- Loads the `.joblib` pipelines saved by Week 5 from `models/`.
- Does **not** recreate preprocessing, `VarianceThreshold`, `SelectKBest`, or classifiers in this notebook.
- Does **not** hardcode or validate against an `expected_steps` list.
- Does **not** refit or retune the Week 5 models.
- Uses each saved pipeline exactly as fitted in Week 5 and applies it to the held-out validation set.
- Records the actual pipeline steps found inside each saved artifact for traceability.
- Saves predicted classes and `predict_proba()` outputs for later Week 6 analyses.

Because the complete fitted Week 5 pipeline is loaded directly, any pipeline structure saved by Week 5 is preserved automatically.

### Required outputs
- `outputs/metrics/validation_results.csv`
- `outputs/metrics/validation_predictions.csv`
- `outputs/metrics/validation_probabilities.csv`
- validation confusion matrices
- `outputs/tables/validation_comparison.csv`
- `outputs/tables/validation_best_models.csv`
- `outputs/tables/week5_model_artifact_audit.csv`
- `outputs/figures/validation_model_comparison.png`


## 1. Setup

In [ ]:
from pathlib import Path
import re
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Run this notebook from the project repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Shared project utilities are still used for feature exclusion
# and metric calculation, but the fitted Week 5 preprocessing
# comes from the saved .joblib pipeline itself.
from src.modeling.preprocessing import prepare_dataset
from src.modeling.evaluation import calculate_metrics

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for folder in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)
print("Week 5 model directory:", MODELS_DIR)
print("Validation mode: load fitted Week 5 .joblib pipelines; no refitting")


## 2. Discover the saved Week 5 tuned pipelines

Notebook 11b saves each fitted `GridSearchCV.best_estimator_` pipeline with `joblib.dump()`. These saved artifacts are the source of truth for validation.

12a therefore discovers the `.joblib` files in `models/` and evaluates those fitted pipelines directly. CSV tuning summaries may still be useful for reporting, but they are no longer needed to reconstruct the models.


In [ ]:
if not MODELS_DIR.exists():
    raise FileNotFoundError(
        f"Week 5 models directory not found: {MODELS_DIR}"
    )

joblib_files = sorted(MODELS_DIR.glob("*.joblib"))

if not joblib_files:
    raise FileNotFoundError(
        "No Week 5 .joblib model artifacts were found in models/. "
        "Copy the fitted pipelines saved by Notebook 11b into this directory."
    )

print(f"Found {len(joblib_files)} Week 5 model artifact(s):")
for path in joblib_files:
    print(" -", path.name)


## 3. Build the saved-model inventory

The dataset and model names are derived from the Week 5 artifact filenames. Both `multimodal_*` and `full_multimodal_*` are treated as the Full Multimodal dataset so the notebook remains compatible with the filenames produced by Week 5.

No hyperparameter dictionary is used to recreate a model here—the fitted `.joblib` pipeline itself is evaluated.


In [ ]:
def clean_text(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


MODEL_SUFFIXES = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}

DATASET_PREFIXES = {
    "demographics_plus_questionnaire": "Demographics + Questionnaire",
    "wearable_plus_questionnaire": "Wearable + Questionnaire",
    "full_multimodal": "Full Multimodal",
    "multimodal": "Full Multimodal",
}


def parse_week5_artifact(path):
    stem = clean_text(path.stem)

    model_key = None
    model_name = None

    for suffix, display_name in MODEL_SUFFIXES.items():
        if stem.endswith(suffix):
            model_key = suffix
            model_name = display_name
            dataset_part = stem[: -(len(suffix) + 1)]
            break

    if model_name is None:
        return None

    dataset_name = DATASET_PREFIXES.get(dataset_part)

    if dataset_name is None:
        return None

    return {
        "dataset": dataset_name,
        "model": model_name,
        "model_key": model_key,
        "filename": path.name,
        "model_path": path,
    }


artifact_records = []

for path in joblib_files:
    record = parse_week5_artifact(path)

    if record is None:
        print(f"⚠ Unrecognized Week 5 artifact filename: {path.name}")
        continue

    artifact_records.append(record)


model_inventory = pd.DataFrame(artifact_records)

if model_inventory.empty:
    raise RuntimeError(
        "No recognized Week 5 tuned pipelines could be mapped from models/."
    )

duplicates = model_inventory.duplicated(
    subset=["dataset", "model"],
    keep=False,
)

if duplicates.any():
    display(model_inventory[duplicates])
    raise ValueError(
        "Duplicate Week 5 artifacts were found for the same dataset/model."
    )

model_inventory = model_inventory.sort_values(
    ["dataset", "model"]
).reset_index(drop=True)

display(model_inventory)
print(f"Recognized fitted Week 5 pipelines: {len(model_inventory)}")


## 4. Find the held-out validation datasets

Only the held-out validation datasets are needed. The Week 5 fitted pipelines are **not retrained** in this notebook.

`prepare_dataset()` is used only to apply the project's standard predictor/target column exclusions to the validation table. Its newly created preprocessing object is ignored because preprocessing is already fitted inside each saved Week 5 pipeline.


In [ ]:
# Optional manual overrides.
# Only fill these in if automatic discovery does not find the correct validation file.
MANUAL_VALIDATION_FILES = {
    # Example:
    # "Full Multimodal": DATA_DIR / "processed" / "multimodal_full_validation.csv",
}

all_csvs = []

for root in [DATA_DIR, OUTPUTS_DIR]:
    if root.exists():
        all_csvs.extend(root.rglob("*.csv"))


def choose_validation_file(dataset):
    if dataset in MANUAL_VALIDATION_FILES:
        path = Path(MANUAL_VALIDATION_FILES[dataset])
        return path if path.exists() else None

    dataset_aliases = {
        "Demographics + Questionnaire": [
            "demographics_questionnaire",
            "demographics_plus_questionnaire",
        ],
        "Wearable + Questionnaire": [
            "wearable_questionnaire",
            "wearable_plus_questionnaire",
        ],
        "Full Multimodal": [
            "multimodal_full",
            "full_multimodal",
            "multimodal",
        ],
    }

    aliases = dataset_aliases.get(
        dataset,
        [clean_text(dataset)],
    )

    candidates = []

    for path in all_csvs:
        name = clean_text(path.stem)

        if not any(
            word in name
            for word in ["validation", "valid", "val"]
        ):
            continue

        if any(
            bad in name
            for bad in [
                "validation_results",
                "validation_predictions",
                "validation_probabilities",
                "confusion_matrix",
                "model_comparison",
                "classification_report",
                "metrics",
                "audit",
            ]
        ):
            continue

        score = 0

        for alias in aliases:
            alias = clean_text(alias)

            if alias in name:
                score += 100

            score += len(
                set(alias.split("_"))
                & set(name.split("_"))
            )

        candidates.append((score, path))

    candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    if candidates and candidates[0][0] > 0:
        return candidates[0][1]

    return None


data_records = []

for dataset in model_inventory["dataset"].unique():
    data_records.append({
        "dataset": dataset,
        "validation_file": choose_validation_file(dataset),
    })

data_inventory = pd.DataFrame(data_records)

display(data_inventory)

missing = data_inventory[
    data_inventory["validation_file"].isna()
]

if not missing.empty:
    print("\nAutomatic discovery could not find every validation file.")
    print("Available CSV files containing 'valid' or 'val':")

    for path in all_csvs:
        name = path.name.lower()
        if "valid" in name or "_val" in name:
            try:
                shown = path.relative_to(PROJECT_ROOT)
            except ValueError:
                shown = path
            print(" -", shown)

    print(
        "\nIf the correct files are listed above, add them to "
        "MANUAL_VALIDATION_FILES and rerun this section."
    )


## 5. Load and inspect the saved Week 5 pipelines

Each `.joblib` artifact is loaded directly with `joblib.load()`.

The notebook records the pipeline steps **as they actually exist inside the saved Week 5 artifact**. It does not compare them with a hardcoded `expected_steps` list. This removes the risk of 12a claiming a match based on an outdated locally defined pipeline order.


In [ ]:
def load_week5_pipeline(model_path):
    model = joblib.load(model_path)

    if not hasattr(model, "predict"):
        raise TypeError(
            f"Saved artifact does not expose predict(): {model_path}"
        )

    return model


def get_pipeline_metadata(model):
    if hasattr(model, "steps"):
        step_names = [
            name
            for name, _ in model.steps
        ]
    else:
        step_names = []

    named_steps = getattr(
        model,
        "named_steps",
        {},
    )

    variance_step = named_steps.get(
        "variance_filter"
    )
    selection_step = named_steps.get(
        "feature_selection"
    )
    classifier_step = named_steps.get(
        "classifier"
    )

    return {
        "pipeline_steps": " -> ".join(step_names),
        "variance_threshold": getattr(
            variance_step,
            "threshold",
            None,
        ),
        "feature_selection__k": getattr(
            selection_step,
            "k",
            None,
        ),
        "classifier_class": (
            type(classifier_step).__name__
            if classifier_step is not None
            else type(model).__name__
        ),
    }


artifact_audit_rows = []

for _, row in model_inventory.iterrows():
    model = load_week5_pipeline(
        row["model_path"]
    )

    metadata = get_pipeline_metadata(model)

    artifact_audit_rows.append({
        "dataset": row["dataset"],
        "model": row["model"],
        "filename": row["filename"],
        **metadata,
        "source": "saved Week 5 fitted pipeline",
        "status": "LOADED",
    })

week5_model_artifact_audit = pd.DataFrame(
    artifact_audit_rows
)

week5_model_artifact_audit.to_csv(
    TABLES_DIR / "week5_model_artifact_audit.csv",
    index=False,
)

display(week5_model_artifact_audit)


## 6. Evaluate the saved Week 5 pipelines on held-out validation

For every saved Week 5 pipeline:

1. load the fitted `.joblib` artifact;
2. load the corresponding held-out validation dataset;
3. prepare the raw predictor columns using the shared project exclusion logic;
4. align columns to `feature_names_in_` when the saved pipeline exposes it;
5. call `predict()` and `predict_proba()` directly.

**No `.fit()` or tuning is performed in 12a.** Preprocessing, variance filtering, feature selection, and classifier parameters/states are all those already fitted and saved by Week 5.


In [ ]:
results = []
prediction_store = {}
prediction_rows = []
probability_rows = []


for _, model_row in model_inventory.iterrows():
    dataset = model_row["dataset"]
    model_name = model_row["model"]
    model_path = model_row["model_path"]

    data_row = data_inventory[
        data_inventory["dataset"] == dataset
    ]

    if data_row.empty:
        print(
            f"✗ {dataset} | {model_name}: "
            "no validation dataset mapping"
        )
        continue

    validation_file = (
        data_row.iloc[0]["validation_file"]
    )

    if pd.isna(validation_file):
        print(
            f"✗ {dataset} | {model_name}: "
            "validation file missing"
        )
        continue

    try:
        pipeline = load_week5_pipeline(
            model_path
        )

        validation_df = pd.read_csv(
            validation_file,
            dtype={ID_COLUMN: str},
        )

        if TARGET not in validation_df.columns:
            raise ValueError(
                f"{TARGET!r} missing from validation data"
            )

        # Use shared project feature/target exclusion only.
        # The returned preprocessing object is intentionally ignored:
        # the saved Week 5 pipeline already contains fitted preprocessing.
        X_val, y_val, _ = prepare_dataset(
            validation_df
        )

        # Align raw validation columns to the exact schema
        # expected by the fitted Week 5 pipeline.
        if hasattr(
            pipeline,
            "feature_names_in_",
        ):
            expected_columns = list(
                pipeline.feature_names_in_
            )

            missing_columns = [
                column
                for column in expected_columns
                if column not in X_val.columns
            ]

            if missing_columns:
                raise ValueError(
                    "Validation data are missing "
                    f"{len(missing_columns)} feature(s) expected "
                    "by the saved Week 5 pipeline: "
                    f"{missing_columns[:10]}"
                )

            X_val = X_val[
                expected_columns
            ].copy()

        # IMPORTANT: no fit() call here.
        y_pred = pipeline.predict(
            X_val
        )

        metrics = calculate_metrics(
            y_val,
            y_pred,
        )

        metadata = get_pipeline_metadata(
            pipeline
        )

        key = (
            f"{clean_text(dataset)}"
            f"__{clean_text(model_name)}"
        )

        prediction_store[key] = (
            y_val.copy(),
            y_pred.copy(),
        )

        results.append({
            "result_key": key,
            "dataset": dataset,
            "model": model_name,
            "artifact_filename": model_row["filename"],
            "n_validation": len(y_val),
            "pipeline_steps": metadata["pipeline_steps"],
            "variance_threshold": metadata["variance_threshold"],
            "feature_selection__k": metadata["feature_selection__k"],
            **metrics,
        })

        if ID_COLUMN in validation_df.columns:
            participant_ids = (
                validation_df[
                    ID_COLUMN
                ].astype(str).values
            )
        else:
            participant_ids = (
                np.arange(
                    len(validation_df)
                ).astype(str)
            )

        for (
            participant_id,
            true_label,
            predicted_label,
        ) in zip(
            participant_ids,
            y_val,
            y_pred,
        ):
            prediction_rows.append({
                ID_COLUMN: participant_id,
                "dataset": dataset,
                "model": model_name,
                "true_label": true_label,
                "predicted_label": predicted_label,
            })

        if hasattr(
            pipeline,
            "predict_proba",
        ):
            probabilities = (
                pipeline.predict_proba(
                    X_val
                )
            )

            classes = getattr(
                pipeline,
                "classes_",
                None,
            )

            if classes is None:
                classifier = getattr(
                    pipeline,
                    "named_steps",
                    {},
                ).get("classifier")

                classes = getattr(
                    classifier,
                    "classes_",
                    np.arange(
                        probabilities.shape[1]
                    ),
                )

            for row_index, (
                participant_id,
                true_label,
            ) in enumerate(
                zip(
                    participant_ids,
                    y_val,
                )
            ):
                probability_row = {
                    ID_COLUMN: participant_id,
                    "dataset": dataset,
                    "model": model_name,
                    "true_label": true_label,
                }

                for (
                    class_index,
                    class_label,
                ) in enumerate(classes):
                    probability_row[
                        f"prob_class_{class_label}"
                    ] = probabilities[
                        row_index,
                        class_index,
                    ]

                probability_rows.append(
                    probability_row
                )

        else:
            print(
                f"⚠ {dataset} | {model_name}: "
                "saved pipeline does not expose predict_proba()"
            )

        print(
            f"✓ {dataset} | {model_name} | "
            f"{model_row['filename']} | "
            f"Macro F1={metrics['macro_f1']:.4f}"
        )

    except Exception as exc:
        print(
            f"✗ {dataset} | {model_name}: "
            f"{type(exc).__name__}: {exc}"
        )


validation_results = pd.DataFrame(
    results
)

if validation_results.empty:
    raise RuntimeError(
        "No saved Week 5 pipeline completed validation successfully."
    )


validation_results = (
    validation_results
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

validation_results.insert(
    0,
    "rank",
    np.arange(
        1,
        len(validation_results) + 1,
    ),
)


validation_predictions = pd.DataFrame(
    prediction_rows
)

validation_probabilities = pd.DataFrame(
    probability_rows
)


validation_predictions.to_csv(
    METRICS_DIR / "validation_predictions.csv",
    index=False,
)

validation_probabilities.to_csv(
    METRICS_DIR / "validation_probabilities.csv",
    index=False,
)


print(
    "Saved class-prediction rows:",
    len(validation_predictions),
)

print(
    "Saved probability rows:",
    len(validation_probabilities),
)

print(
    "Saved Week 5 pipelines evaluated:",
    len(validation_results),
)

display(validation_results)


## 7. Save `validation_results.csv` and comparison table

In [ ]:
validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

comparison_cols = [
    "rank",
    "dataset",
    "model",
    "n_validation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_comparison = validation_results[
    comparison_cols
].copy()

validation_comparison.to_csv(
    TABLES_DIR / "validation_comparison.csv",
    index=False,
)

display(validation_comparison.round(4))

## 8. Validation confusion matrices

In [ ]:
# Generate and save validation confusion matrices as tables

for _, row in validation_results.iterrows():
    key = row["result_key"]
    y_true, y_pred = prediction_store[key]

    labels = sorted(
        pd.Series(y_true).dropna().unique().tolist()
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual {x}" for x in labels],
        columns=[f"Predicted {x}" for x in labels]
    )

    # Save confusion matrix
    csv_path = (
        METRICS_DIR /
        f"{key}_validation_confusion_matrix.csv"
    )

    cm_df.to_csv(csv_path)

    # Show it directly in the notebook
    print(f"\n{row['dataset']} — {row['model']}")
    display(cm_df)

    print(f"Saved: {csv_path.name}")

## 9. Validation comparison figure

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

plot_df = validation_results[
    ["dataset", "model", "macro_f1", "balanced_accuracy", "accuracy"]
].copy()

plot_df["candidate"] = (
    plot_df["dataset"].astype(str)
    + " | "
    + plot_df["model"].astype(str)
)

plot_df = plot_df.sort_values(
    ["macro_f1", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

figure_path = Path(FIGURES_DIR) / "validation_model_comparison.png"

img = Image.new("RGB", (1200, 850), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default()

draw.text((40, 30), "Validation Model Comparison", fill="black", font=font)
draw.text(
    (40, 55),
    "Models ranked by validation Macro F1. Higher scores are better.",
    fill="black",
    font=font
)

bar_x = 430
bar_width = 650

for i, row in plot_df.iterrows():
    y = 110 + i * 110

    label = f"#{i+1} {row['candidate']}"
    if i == 0:
        label += " - BEST"

    draw.text((40, y), label, fill="black", font=font)

    draw.rectangle(
        [bar_x, y + 25, bar_x + bar_width, y + 50],
        fill="lightgray"
    )

    score_width = int(float(row["macro_f1"]) * bar_width)

    draw.rectangle(
        [bar_x, y + 25, bar_x + score_width, y + 50],
        fill="steelblue"
    )

    metrics = (
        f"Macro F1: {row['macro_f1']:.4f} | "
        f"Balanced Acc: {row['balanced_accuracy']:.4f} | "
        f"Accuracy: {row['accuracy']:.4f}"
    )

    draw.text((40, y + 60), metrics, fill="black", font=font)

img.save(str(figure_path), format="PNG")

print("Saved:", figure_path)
print("Exists:", figure_path.exists())

## 10. Select the best candidate model(s)

In [ ]:
best_by_dataset = (
    validation_results
    .sort_values(
        [
            "dataset",
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[True, False, False],
    )
    .groupby(
        "dataset",
        as_index=False,
    )
    .first()
)

best_by_dataset.to_csv(
    TABLES_DIR / "validation_best_models.csv",
    index=False,
)

display(
    best_by_dataset[
        [
            "dataset",
            "model",
            "macro_f1",
            "balanced_accuracy",
            "accuracy",
        ]
    ].round(4)
)

best = validation_results.iloc[0]

print(
    f"Best overall candidate: "
    f"{best['model']} ({best['dataset']})"
)
print(
    f"Macro F1: {best['macro_f1']:.4f}"
)
print(
    f"Balanced accuracy: "
    f"{best['balanced_accuracy']:.4f}"
)

## 11. Validation metrics summary and justification

In [ ]:
display(
    validation_results[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

print("\nJUSTIFICATION")
print(
    f"{best['model']} on {best['dataset']} "
    f"is the strongest candidate to move forward "
    f"because it achieved the highest validation "
    f"Macro F1 ({best['macro_f1']:.4f}). "
    f"Its balanced accuracy was "
    f"{best['balanced_accuracy']:.4f}. "
    f"Macro F1 is used as the primary selection "
    f"metric because it gives equal importance to "
    f"performance across classes, while balanced "
    f"accuracy is used as the secondary comparison."
)

## 12. Deliverables Check

In [ ]:
from pathlib import Path
import pandas as pd

# Required validation deliverables
deliverables = pd.DataFrame([
    {
        "deliverable": "Validation results",
        "path": METRICS_DIR / "validation_results.csv",
    },
    {
        "deliverable": "Validation class predictions",
        "path": METRICS_DIR / "validation_predictions.csv",
    },
    {
        "deliverable": "Validation probability predictions",
        "path": METRICS_DIR / "validation_probabilities.csv",
    },
    {
        "deliverable": "Week 5 saved-model artifact audit",
        "path": TABLES_DIR / "week5_model_artifact_audit.csv",
    },
    {
        "deliverable": "Validation comparison table",
        "path": TABLES_DIR / "validation_comparison.csv",
    },
    {
        "deliverable": "Best candidate table",
        "path": TABLES_DIR / "validation_best_models.csv",
    },
    {
        "deliverable": "Validation comparison figure",
        "path": FIGURES_DIR / "validation_model_comparison.png",
    },
])

deliverables["status"] = deliverables["path"].apply(
    lambda p: "READY" if Path(p).exists() else "MISSING"
)

display(deliverables)

confusion_matrices = list(
    METRICS_DIR.glob("*_validation_confusion_matrix.csv")
)

print(
    f"Validation confusion matrices saved: "
    f"{len(confusion_matrices)}"
)

print(
    "Probability rows saved:",
    len(validation_probabilities),
)

if (deliverables["status"] == "READY").all():
    print()
    print("✓ All required validation outputs are ready.")
else:
    print()
    print("⚠ Some required validation outputs are missing.")